# Amyloid ranking — Aβ42 variants

Collects results from 4 tools (TANGO, PASTA, AmyPred-FRL, CrossBeta), calculates consensus rank, outputs top-10 aggregators and top-10 disruptors.

In [ ]:
import pandas as pd
import numpy as np
import io
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.ticker import MaxNLocator

DATA = "../predictions/"

# ── Palette ───────────────────────────────────────────────────────────────────
BG = "#0a0c14"
SURFACE = "#111520"
MUTED = "#64748b"
TEXT = "#e2e8f0"
WT_COL = "#4ff7c0"
AGG_COL = "#ef4444"  # red — increased aggregation
DIS_COL = "#3b82f6"  # blue  — decreased aggregation
ACC_COL = "#fbbf24"  # yellow — accent / consensus

TOOL_COLORS = {
    "TANGO": "#ef4444",
    "PASTA": "#3b82f6",
    "AmyPred-FRL": "#a78bfa",
    "CrossBeta": "#4ff7c0",
}

In [ ]:
# ── Loading ──────────────────────────────────────────────────────────────────


def load_tango(path):
    df = pd.read_csv(path, sep="\t")
    wt = df.loc[df["Sequence"] == "Wildtype_Abeta_42", "Aggregation"].iloc[0]
    rows = []
    for _, r in df.iterrows():
        if "Wildtype" in r["Sequence"]:
            continue
        rows.append({"variant": r["Sequence"], "TANGO": r["Aggregation"] - wt})
    return pd.DataFrame(rows).set_index("variant")


def load_pasta(path):
    with open(path, encoding="utf-8-sig") as f:
        df = pd.read_csv(io.StringIO(f.read().replace(",", ".")), sep=";")
    wt = df.loc[df["Protein name"].str.contains("Wildtype"), "Best Energy"].iloc[0]
    rows = []
    for _, r in df.iterrows():
        if "Wildtype" in r["Protein name"]:
            continue
        rows.append({"variant": r["Protein name"], "PASTA": r["Best Energy"] - wt})
    return pd.DataFrame(rows).set_index("variant")


def load_amypred(path):
    lines = open(path, encoding="utf-8-sig").readlines()
    wt_val, rows = None, []
    for line in lines[1:]:
        line = line.strip()
        if not line:
            continue
        parts = line.split(";")
        name, prob = parts[0], float(parts[2])
        if "Wildtype" in name:
            wt_val = prob
        else:
            rows.append({"variant": name, "AmyPred-FRL": prob})
    df = pd.DataFrame(rows).set_index("variant")
    if wt_val:
        df["AmyPred-FRL"] -= wt_val
    return df


def load_crossbeta(path):
    lines = open(path).readlines()
    wt_val, rows = None, []
    for line in lines[1:]:
        line = line.strip()
        if not line:
            continue
        parts = line.split(";")
        name, avg = parts[0], float(parts[2])
        if "Wildtype" in name:
            wt_val = avg
        else:
            rows.append({"variant": name, "CrossBeta": avg})
    df = pd.DataFrame(rows).set_index("variant")
    if wt_val:
        df["CrossBeta"] -= wt_val
    return df


def load_amylogram(path):
    df = pd.read_csv(path)
    wt_val = df.loc[
        df["Input name"] == "Wildtype_Abeta_42", "Amyloid probability"
    ].iloc[0]
    rows = []
    for _, r in df.iterrows():
        if "Wildtype" in r["Input name"]:
            continue
        rows.append(
            {"variant": r["Input name"], "AmyloGram": r["Amyloid probability"] - wt_val}
        )
    return pd.DataFrame(rows).set_index("variant")

In [ ]:
# ── Build matrix ────────────────────────────────────────────────────────────
tango = load_tango(DATA + "tango/tango_input_aggregation.txt")
pasta = load_pasta(DATA + "pasta/pasta.csv")
amypred = load_amypred(DATA + "amypred/amypred-frl.csv")
crossbeta = load_crossbeta(DATA + "cross-beta/cross-beta_result.csv")

raw = (
    tango.join(pasta, how="outer")
    .join(amypred, how="outer")
    .join(crossbeta, how="outer")
)

raw = raw.dropna()
n = len(raw)
print(f"Variants: {n}")
print(f"Tools: {raw.columns.tolist()}\n")

In [ ]:
# ── Z-score normalization — each tool separately ────────────────────────────
# PASTA: negative delta = lower energy = stronger aggregation → invert
tools = ["TANGO", "PASTA", "AmyPred-FRL", "CrossBeta"]
raw_norm = raw.copy()
raw_norm["PASTA"] = -raw_norm["PASTA"]  # invert: now + = stronger aggregation

zscore = pd.DataFrame(index=raw_norm.index)
for tool in tools:
    zscore[tool] = (raw_norm[tool] - raw_norm[tool].mean()) / raw_norm[tool].std()

# ── Top-10 per tool ───────────────────────────────────────────────────────────
top10_agg_per_tool = {t: zscore[t].nlargest(10).index for t in tools}
top10_dis_per_tool = {t: zscore[t].nsmallest(10).index for t in tools}

# ── Consensus score = mean z-score (for overview plot) ──────────────────────
raw["consensus_z"] = zscore.mean(axis=1)
raw_sorted = raw.sort_values("consensus_z", ascending=False)


def short(name):
    return name.replace("_Abeta42", "").replace("_Abeta_42", "")


# ── Text output ──────────────────────────────────────────────────────────────
for tool in tools:
    print(f"\n{'═' * 55}")
    print(f"  {tool}  —  TOP-10 AGGREGATORS")
    print(f"{'═' * 55}")
    for i, idx in enumerate(top10_agg_per_tool[tool], 1):
        print(
            f"  {i:2}. {short(idx):<22}  z={zscore.loc[idx, tool]:+.3f}  raw={raw.loc[idx, tool]:+.4f}"
        )
    print(f"\n  {tool}  —  TOP-10 DISRUPTORS")
    print(f"{'─' * 55}")
    for i, idx in enumerate(top10_dis_per_tool[tool], 1):
        print(
            f"  {i:2}. {short(idx):<22}  z={zscore.loc[idx, tool]:+.3f}  raw={raw.loc[idx, tool]:+.4f}"
        )


# ── Visualization ────────────────────────────────────────────────────────────
def draw_top10(ax, idx_list, tool, direction, color):
    ax.set_facecolor(SURFACE)
    vals = zscore.loc[idx_list, tool].values
    names = [short(n) for n in idx_list]
    order = np.argsort(vals)[::-1] if direction == "agg" else np.argsort(vals)
    vals = vals[order]
    names = [names[i] for i in order]
    y = np.arange(len(vals))
    ax.barh(y, vals, color=color, alpha=0.25, height=0.55, zorder=1)
    ax.hlines(y, 0, vals, color=color, lw=1.2, alpha=0.5, zorder=2)
    ax.scatter(vals, y, color=color, s=45, zorder=3, linewidths=0)
    for i, v in enumerate(vals):
        ha = "left" if v >= 0 else "right"
        off = 0.05 if v >= 0 else -0.05
        ax.text(
            v + off,
            i,
            f"{v:+.2f}",
            ha=ha,
            va="center",
            fontsize=6.5,
            fontfamily="monospace",
            color=TEXT,
            alpha=0.85,
        )
    ax.axvline(0, color=WT_COL, lw=0.8, linestyle="--", alpha=0.4, zorder=1)
    ax.set_yticks(y)
    ax.set_yticklabels(names, fontsize=8, fontfamily="monospace", color=TEXT)
    ax.tick_params(axis="y", length=0, pad=4)
    ax.tick_params(axis="x", colors=MUTED, labelsize=7)
    for sp in ax.spines.values():
        sp.set_edgecolor("#1e2540")
    ax.set_title(
        tool,
        color=TOOL_COLORS[tool],
        fontsize=10,
        fontfamily="monospace",
        fontweight="bold",
        pad=8,
    )
    ax.set_xlabel(
        "z-score (within tool)",
        color=MUTED,
        fontsize=7.5,
        fontfamily="monospace",
        labelpad=4,
    )
    ax.xaxis.set_major_locator(MaxNLocator(5))

In [ ]:
# Figure 1 — Consensus overview (all 65 variants)

fig1, ax0 = plt.subplots(figsize=(20, 7), facecolor=BG)
ax0.set_facecolor(SURFACE)

cons_sorted = raw["consensus_z"].sort_values(ascending=False)
labels = [short(n) for n in cons_sorted.index]
colors_bar = [AGG_COL if v >= 0 else DIS_COL for v in cons_sorted.values]

bars = ax0.bar(
    range(len(cons_sorted)), cons_sorted.values, color=colors_bar, width=0.75, zorder=2
)
for i, v in enumerate(cons_sorted.values):
    if i < 10 or i >= len(cons_sorted) - 10:
        bars[i].set_edgecolor(ACC_COL)
        bars[i].set_linewidth(1.2)
    else:
        bars[i].set_alpha(0.6)

ax0.axhline(0, color=WT_COL, lw=0.8, linestyle="--", alpha=0.5, zorder=1)
for i, (label, v) in enumerate(zip(labels, cons_sorted.values)):
    if i < 10 or i >= len(cons_sorted) - 10:
        va = "bottom" if v >= 0 else "top"
        off = 0.03 if v >= 0 else -0.03
        ax0.text(
            i,
            v + off,
            label,
            ha="center",
            va=va,
            fontsize=6,
            fontfamily="monospace",
            color=ACC_COL,
            rotation=90,
            fontweight="bold",
        )

ax0.set_xlim(-0.8, len(cons_sorted) - 0.2)
ax0.set_xticks([])
ax0.set_ylabel(
    "consensus z-score (mean across tools)",
    color=MUTED,
    fontsize=9,
    fontfamily="monospace",
    labelpad=6,
)
ax0.tick_params(axis="y", colors=MUTED, labelsize=8)
for sp in ax0.spines.values():
    sp.set_edgecolor("#1e2540")
ax0.set_title(
    "Aβ42  ·  Consensus Ranking  ·  all 65 variants\n"
    "consensus = mean z-score across tools  ·  highlighted top-10 aggregators and top-10 disruptors",
    color=TEXT,
    fontsize=11,
    fontfamily="monospace",
    fontweight="bold",
    pad=10,
)

agg_p = mpatches.Patch(color=AGG_COL, label="Increased aggregation")
dis_p = mpatches.Patch(color=DIS_COL, label="Decreased aggregation")
top_p = mpatches.Patch(
    facecolor="none", edgecolor=ACC_COL, lw=1.5, label="Top-10 (each direction)"
)
ax0.legend(
    handles=[agg_p, dis_p, top_p],
    loc="upper right",
    frameon=True,
    framealpha=0.2,
    edgecolor=MUTED,
    facecolor=BG,
    fontsize=8.5,
    labelcolor=TEXT,
)

fig1.tight_layout(pad=1.5)
fig1.show()
fig1.savefig(
    "ranking_consensus.png",
    dpi=170,
    bbox_inches="tight",
    facecolor=BG,
    edgecolor="none",
)
print("Saved to ranking_consensus.png")
plt.close(fig1)

In [ ]:
# Figure 2 — Top-10 aggregators per tool

fig2, axes2 = plt.subplots(1, 4, figsize=(20, 7), facecolor=BG)
fig2.subplots_adjust(wspace=0.42, left=0.07, right=0.97, top=0.84, bottom=0.1)
fig2.suptitle(
    "Aβ42  ·  Top-10 aggregators  ·  z-score within each tool\n"
    "red = increased aggregation vs WT",
    color=AGG_COL,
    fontsize=11,
    fontfamily="monospace",
    fontweight="bold",
    y=0.97,
)

for ax, tool in zip(axes2, tools):
    draw_top10(ax, top10_agg_per_tool[tool], tool, "agg", TOOL_COLORS[tool])

fig2.show()
fig2.savefig(
    "ranking_top10_aggregators.png",
    dpi=170,
    bbox_inches="tight",
    facecolor=BG,
    edgecolor="none",
)
print("Saved to ranking_top10_aggregators.png")
plt.close(fig2)

In [ ]:
# Figure 3 — Top-10 disruptors per tool

fig3, axes3 = plt.subplots(1, 4, figsize=(20, 7), facecolor=BG)
fig3.subplots_adjust(wspace=0.42, left=0.07, right=0.97, top=0.84, bottom=0.1)
fig3.suptitle(
    "Aβ42  ·  Top-10 disruptors  ·  z-score within each tool\n"
    "blue = decreased aggregation vs WT",
    color=DIS_COL,
    fontsize=11,
    fontfamily="monospace",
    fontweight="bold",
    y=0.97,
)

for ax, tool in zip(axes3, tools):
    draw_top10(ax, top10_dis_per_tool[tool], tool, "dis", TOOL_COLORS[tool])

fig3.savefig(
    "ranking_top10_disruptors.png",
    dpi=170,
    bbox_inches="tight",
    facecolor=BG,
    edgecolor="none",
)
fig3.show()
print("Saved to ranking_top10_disruptors.png")
plt.close(fig3)